In [ ]:
import json, os

path = '/content/drive/MyDrive/project/output/quotations_raw.json'
with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(json.dumps(data[0], indent=2, ensure_ascii=False))

{
  "quotation_id": "QTN-2026-001",
  "quotation_date": "10-Jul-2026",
  "customer": "ABC Manufacturing Pvt. Ltd.",
  "material": "SS304 Stainless Steel Pipe",
  "supplier": null,
  "quantity": null,
  "unit": null,
  "unit_price": null,
  "currency": "INR",
  "delivery_time": "10 Days",
  "source_file": "Quotation 1 (3).pdf"
}


In [8]:
# ============================================================
# Notebook 4 — Clean & Validate (Rewritten: Standardize → Validate → Clean → Merge → Export)
# File: 04_clean_validate.ipynb
# ============================================================

# --- Cell 1: Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/project'
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'output')

# --- Cell 2: Imports ---
import json
import re
from datetime import datetime, timezone

# --- Cell 3: Load raw datasets ---
def load_json(filename):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

raw_quotations = load_json('quotations_raw.json')
raw_supplier_prices = load_json('supplier_prices_raw.json')
raw_exchange_rates = load_json('exchange_rates.json')

print(f"Loaded {len(raw_supplier_prices)} supplier rows, "
      f"{len(raw_quotations)} quotations, "
      f"{len(raw_exchange_rates)} exchange rates")


# ============================================================
# STEP 5 (moved first) — STANDARDIZE
# Rename each source's raw field names to the unified schema
# BEFORE validation/cleaning ever runs, so downstream steps only
# ever have to know one set of key names.
# ============================================================

SUPPLIER_FIELD_MAP = {
    'Material': 'material',
    'Specification': 'specification',
    'Supplier': 'supplier',
    'Unit Price': 'unit_price',
    'Currency': 'currency',
    'Unit': 'unit',
    'Lead Time (Days)': 'lead_time',
    'MOQ': 'moq',
    'Available Stock': 'available_stock',
}

QUOTATION_FIELD_MAP = {
    'quotation_id': 'quotation_id',
    'quotation_date': 'quotation_date',
    'customer': 'customer',
    'material': 'material',
    'supplier': 'supplier',
    'quantity': 'quantity',
    'unit': 'unit',
    'unit_price': 'unit_price',
    'currency': 'currency',
    'delivery_time': 'delivery_time',
    'source_file': 'source_file',
}

RATE_FIELD_MAP = {
    'base_currency': 'base_currency',
    'target_currency': 'target_currency',
    'exchange_rate': 'exchange_rate',
    'date': 'date',
}

def standardize_record(raw_record, field_map):
    """Renames raw source keys to the unified schema. Missing source
    keys become None rather than raising, so partial records surface
    as validation failures instead of crashing the pipeline."""
    return {std_key: raw_record.get(raw_key) for raw_key, std_key in field_map.items()}

standardized_quotations = [standardize_record(q, QUOTATION_FIELD_MAP) for q in raw_quotations]
standardized_supplier_prices = [standardize_record(r, SUPPLIER_FIELD_MAP) for r in raw_supplier_prices]
standardized_exchange_rates = [standardize_record(r, RATE_FIELD_MAP) for r in raw_exchange_rates]

print("\nSample standardized quotation:")
print(json.dumps(standardized_quotations[0], indent=2, ensure_ascii=False))
print("\nSample standardized supplier row:")
print(json.dumps(standardized_supplier_prices[0], indent=2, ensure_ascii=False))


# ============================================================
# STEP 3 — VALIDATE
# Runs against standardized data, so key names are guaranteed
# to match regardless of source format quirks.
# ============================================================

def validate_quotation(q):
    errors = []
    if not q.get('customer'):
        errors.append('missing customer')
    if not q.get('material'):
        errors.append('missing material')
    price = q.get('unit_price')
    if price is None or not isinstance(price, (int, float)) or price <= 0:
        errors.append('invalid unit_price')
    qty = q.get('quantity')
    if qty is None or not isinstance(qty, (int, float)) or qty <= 0:
        errors.append('invalid quantity')
    return errors

def validate_supplier_row(row):
    errors = []
    if not row.get('supplier'):
        errors.append('missing supplier')
    currency = row.get('currency')
    if not currency or currency not in ('INR', 'USD', 'EUR', 'GBP'):
        errors.append('invalid currency')
    price = row.get('unit_price')
    if price is None or not isinstance(price, (int, float)) or price <= 0:
        errors.append('invalid unit_price')
    return errors

def validate_exchange_rate(rate):
    errors = []
    value = rate.get('exchange_rate')
    if value is None:
        errors.append('missing rate')
    elif not isinstance(value, (int, float)):
        errors.append('rate not numeric')
    elif value <= 0:
        errors.append('rate not positive')
    return errors


validation_report = {'quotations': [], 'supplier_prices': [], 'exchange_rates': []}

for i, q in enumerate(standardized_quotations):
    errs = validate_quotation(q)
    validation_report['quotations'].append({
        'index': i,
        'id': q.get('quotation_id', f'row_{i}'),
        'valid': len(errs) == 0,
        'errors': errs
    })

for i, row in enumerate(standardized_supplier_prices):
    errs = validate_supplier_row(row)
    validation_report['supplier_prices'].append({
        'index': i,
        'id': row.get('material', f'row_{i}'),
        'valid': len(errs) == 0,
        'errors': errs
    })

for i, rate in enumerate(standardized_exchange_rates):
    errs = validate_exchange_rate(rate)
    validation_report['exchange_rates'].append({
        'index': i,
        'id': rate.get('target_currency', f'row_{i}'),
        'valid': len(errs) == 0,
        'errors': errs
    })

for dataset_name, results in validation_report.items():
    total = len(results)
    passed = sum(1 for r in results if r['valid'])
    print(f"\n{dataset_name}: {passed}/{total} passed")
    for r in results:
        if not r['valid']:
            print(f"  ❌ {r['id']}: {r['errors']}")

with open(os.path.join(OUTPUT_DIR, 'validation_report.json'), 'w', encoding='utf-8') as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)


# ============================================================
# STEP 4 — CLEAN
# Only cleans records that passed validation; invalid records are
# excluded rather than force-cleaned, so clean_data.json stays trustworthy.
# ============================================================

def clean_currency_value(value):
    """₹820 / 820 INR / Rs.820 / 820 -> 820.0 (float)"""
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    s = str(value)
    s = re.sub(r'[₹$€£]', '', s)
    s = re.sub(r'(?i)\b(rs\.?|inr|usd|eur|gbp)\b', '', s)
    s = s.replace(',', '').strip()
    match = re.search(r'-?\d+\.?\d*', s)
    return float(match.group()) if match else None

SUPPLIER_NAME_MAP = {
    'tata': 'Tata Steel',
    'tatasteel': 'Tata Steel',
    'tata steel ltd': 'Tata Steel',
    'tata steel': 'Tata Steel',
    'hindalco': 'Hindalco',
    'hindalco industries': 'Hindalco',
    'vedanta': 'Vedanta',
    'jsw': 'JSW Steel',
    'jsw steel': 'JSW Steel',
    'bharat metals': 'Bharat Metals',
    'sail': 'SAIL',
    'jindal': 'Jindal Stainless',
    'jindal stainless': 'Jindal Stainless',
}

def clean_supplier_name(name):
    if not name:
        return None
    key = re.sub(r'[^a-z0-9\s]', '', name.lower()).strip()
    key = re.sub(r'\s+', ' ', key)
    return SUPPLIER_NAME_MAP.get(key, name.strip().title())

def clean_date(date_str):
    """Handles: '20 Jul 2026', '20/07/26', '2026-07-20', '10-Jul-2026' -> '2026-07-20'"""
    if not date_str:
        return None
    date_str = date_str.strip()
    formats = [
        '%d %b %Y', '%d-%b-%Y', '%d/%m/%y', '%d/%m/%Y',
        '%Y-%m-%d', '%d-%m-%Y', '%d %B %Y',
    ]
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return None  # unparseable — flagged, not guessed

def clean_quotation(q):
    return {
        'quotation_id': q.get('quotation_id'),
        'quotation_date': clean_date(q.get('quotation_date')),
        'customer': q.get('customer', '').strip() if q.get('customer') else None,
        'material': q.get('material', '').strip() if q.get('material') else None,
        'supplier': clean_supplier_name(q.get('supplier')),
        'quantity': q.get('quantity'),
        'unit': q.get('unit'),
        'unit_price': clean_currency_value(q.get('unit_price')),
        'currency': q.get('currency', 'INR'),
        'delivery_time': q.get('delivery_time'),
        'source_file': q.get('source_file'),
    }

def clean_supplier_row(row):
    return {
        'material': row.get('material', '').strip() if row.get('material') else None,
        'specification': row.get('specification'),
        'supplier': clean_supplier_name(row.get('supplier')),
        'unit_price': clean_currency_value(row.get('unit_price')),
        'currency': row.get('currency', 'INR'),
        'unit': row.get('unit'),
        'lead_time': row.get('lead_time'),
        'moq': row.get('moq'),
        'available_stock': row.get('available_stock'),
    }

def clean_exchange_rate(rate):
    return {
        'base_currency': rate.get('base_currency'),
        'target_currency': rate.get('target_currency'),
        'exchange_rate': float(rate['exchange_rate']) if rate.get('exchange_rate') is not None else None,
        'date': clean_date(rate.get('date')) or rate.get('date'),
    }

valid_quotation_indices = {r['index'] for r in validation_report['quotations'] if r['valid']}
valid_supplier_indices = {r['index'] for r in validation_report['supplier_prices'] if r['valid']}
valid_rate_indices = {r['index'] for r in validation_report['exchange_rates'] if r['valid']}

cleaned_quotations = [clean_quotation(q) for i, q in enumerate(standardized_quotations) if i in valid_quotation_indices]
cleaned_supplier_prices = [clean_supplier_row(r) for i, r in enumerate(standardized_supplier_prices) if i in valid_supplier_indices]
cleaned_exchange_rates = [clean_exchange_rate(r) for i, r in enumerate(standardized_exchange_rates) if i in valid_rate_indices]

print(f"\nCleaned: {len(cleaned_quotations)}/{len(standardized_quotations)} quotations, "
      f"{len(cleaned_supplier_prices)}/{len(standardized_supplier_prices)} supplier rows, "
      f"{len(cleaned_exchange_rates)}/{len(standardized_exchange_rates)} exchange rates")


# ============================================================
# STEP 6 — MERGE
# ============================================================

rate_lookup = {r['target_currency']: r['exchange_rate'] for r in cleaned_exchange_rates}

def attach_inr_value(record):
    """Adds unit_price_inr using live rates, so all records are comparable regardless of source currency."""
    currency = record.get('currency', 'INR')
    price = record.get('unit_price')
    if price is None:
        record['unit_price_inr'] = None
    elif currency == 'INR':
        record['unit_price_inr'] = price
    elif currency in rate_lookup:
        record['unit_price_inr'] = round(price * rate_lookup[currency], 2)
    else:
        record['unit_price_inr'] = None  # rate unavailable — flagged, not guessed
    return record

cleaned_quotations = [attach_inr_value(q) for q in cleaned_quotations]
cleaned_supplier_prices = [attach_inr_value(r) for r in cleaned_supplier_prices]

unified_dataset = {
    'quotations': cleaned_quotations,
    'supplier_prices': cleaned_supplier_prices,
    'exchange_rates': cleaned_exchange_rates,
    'metadata': {
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'quotations_count': len(cleaned_quotations),
        'supplier_prices_count': len(cleaned_supplier_prices),
        'exchange_rates_count': len(cleaned_exchange_rates),
        'quotations_dropped_invalid': len(standardized_quotations) - len(cleaned_quotations),
        'supplier_rows_dropped_invalid': len(standardized_supplier_prices) - len(cleaned_supplier_prices),
    }
}


# ============================================================
# STEP 7 — EXPORT
# ============================================================

output_path = os.path.join(OUTPUT_DIR, 'clean_data.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(unified_dataset, f, indent=2, ensure_ascii=False)

print(f"\n✅ Exported unified dataset to {output_path}")
print(json.dumps(unified_dataset['metadata'], indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 30 supplier rows, 8 quotations, 3 exchange rates

Sample standardized quotation:
{
  "quotation_id": "QTN-2026-001",
  "quotation_date": "10-Jul-2026",
  "customer": "ABC Manufacturing Pvt. Ltd.",
  "material": "SS304 Stainless Steel Pipe",
  "supplier": "Tata Steel",
  "quantity": 150,
  "unit": "kg",
  "unit_price": 820,
  "currency": "INR",
  "delivery_time": "10 Days",
  "source_file": "Quotation 1 (3).pdf"
}

Sample standardized supplier row:
{
  "material": "SS304 Stainless Steel Pipe",
  "specification": "ASTM A312",
  "supplier": "Tata Steel",
  "unit_price": 820,
  "currency": "INR",
  "unit": "kg",
  "lead_time": 10,
  "moq": 100,
  "available_stock": 5000
}

quotations: 8/8 passed

supplier_prices: 30/30 passed

exchange_rates: 3/3 passed

Cleaned: 8/8 quotations, 30/30 supplier rows, 3/3 exchange rates

✅ Exported unified dataset to /conten